# Test: Quality Cleaner Agent

Tests the quality-cleaner agent on a real content file from the monorepo.
No content is modified — this is read-only exploration.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv('../.env')

from openai import OpenAI
from agent_loader import load_agent
from openai_caller import call_agent

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
agent = load_agent(Path('../agents/01-quality-cleaner.md'))
print(f'Loaded agent: {agent.name} (model={agent.model}, temp={agent.temperature})')

In [ ]:
sample_file = Path('../../apps/deck-repair/src/data/generated_content/how_to_choose_the_best_deck_repair_contractor_in_portland.md')
sample = sample_file.read_text(encoding='utf-8')
print(f'File: {sample_file.name}')
print(f'Length: {len(sample)} chars')
print('\n--- First 1000 chars ---')
print(sample[:1000])

In [ ]:
result = call_agent(
    client=client,
    model=agent.model,
    temperature=agent.temperature,
    system_prompt=agent.system_prompt,
    content=sample,
)
print(f'Result length: {len(result)} chars')
print('\n--- First 1000 chars after quality cleaner ---')
print(result[:1000])

In [ ]:
import re
citation_pattern = r'\[Source:[^\]]+\]'
before_count = len(re.findall(citation_pattern, sample))
after_count = len(re.findall(citation_pattern, result))
print(f'Citation artifacts: {before_count} -> {after_count}')

# Check for malformed link injection pattern
malformed_pattern = r'For (related services|comprehensive solutions),'
before_malformed = len(re.findall(malformed_pattern, sample))
after_malformed = len(re.findall(malformed_pattern, result))
print(f'Malformed link injections: {before_malformed} -> {after_malformed}')

print(f'\nLength change: {len(sample)} -> {len(result)} ({len(result)-len(sample):+d} chars)')